# 09 — Fine-tune Error Analysis

Our fine-tuned Whisper model achieves 48% exact match on Dothraki transcription — better than the phoneme baseline but far behind DTW (79.5%) and embedding (73.5%). This notebook dissects the other 52%: where does the model fail, and what patterns emerge in its errors?

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / 'data' / 'results'

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

C_TEAL = '#4ecdc4'
C_RED = '#ff6b6b'
C_YELLOW = '#ffd93d'
C_DARK_TEAL = '#45b7aa'
COLORS = [C_TEAL, C_RED, C_YELLOW, C_DARK_TEAL]

# Load fine-tune results
ft_data = json.loads((RESULTS_DIR / 'batch_eval_finetune_small.json').read_text())
results = ft_data['results']

print(f'Total clips: {len(results)}')
print(f'Exact matches: {ft_data["stats"]["exact_matches"]} ({ft_data["stats"]["exact_match_rate"]:.1%})')

---
## 1. Error Categorization

Classify each non-exact-match into categories: prefix match (starts correctly but over-generates), empty output, or entirely wrong.

In [ ]:
categories = {'exact': 0, 'prefix_match': 0, 'empty': 0, 'wrong': 0}
categorized = []

for r in results:
    gt = r['gt_dothraki'].strip().lower()
    output = (r.get('raw_dothraki') or '').strip().lower()
    
    if r.get('exact_match', False):
        cat = 'exact'
    elif not output:
        cat = 'empty'
    elif output.startswith(gt[:min(len(gt), 10)]) or gt.startswith(output[:min(len(output), 10)]):
        cat = 'prefix_match'
    else:
        cat = 'wrong'
    
    categories[cat] += 1
    categorized.append({**r, 'category': cat, 'gt_lower': gt, 'out_lower': output})

fig, ax = plt.subplots(figsize=(10, 6))
cat_names = list(categories.keys())
cat_counts = list(categories.values())
cat_colors = [C_TEAL, C_YELLOW, C_RED, '#ff4444']

bars = ax.bar(cat_names, cat_counts, color=cat_colors, edgecolor='#1a1a2e', alpha=0.85)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=12)
ax.set_xlabel('Error Category')
ax.set_ylabel('Number of Clips')
ax.set_title('Fine-tune Output Categorization (200 clips)')
plt.tight_layout()
plt.show()

for cat, count in categories.items():
    print(f'{cat}: {count} ({count/len(results):.1%})')

---
## 2. Levenshtein Distance Distribution

Character-level edit distance between ground truth and model output — how "close" are the errors?

In [ ]:
def levenshtein(s1, s2):
    """Compute character-level Levenshtein distance."""
    if len(s1) < len(s2):
        return levenshtein(s2, s1)
    if len(s2) == 0:
        return len(s1)
    prev_row = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        curr_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = prev_row[j + 1] + 1
            deletions = curr_row[j] + 1
            substitutions = prev_row[j] + (c1 != c2)
            curr_row.append(min(insertions, deletions, substitutions))
        prev_row = curr_row
    return prev_row[-1]

distances = []
for r in categorized:
    dist = levenshtein(r['gt_lower'], r['out_lower'])
    distances.append(dist)

distances = np.array(distances)

fig, ax = plt.subplots(figsize=(14, 6))
ax.hist(distances, bins=30, color=C_TEAL, edgecolor='#1a1a2e', alpha=0.85)
ax.axvline(np.mean(distances), color=C_RED, linestyle='--', linewidth=2, label=f'Mean: {np.mean(distances):.1f}')
ax.axvline(np.median(distances), color=C_DARK_TEAL, linestyle=':', linewidth=2, label=f'Median: {np.median(distances):.1f}')
ax.set_xlabel('Levenshtein Distance (characters)')
ax.set_ylabel('Count')
ax.set_title('Edit Distance Distribution: Ground Truth vs Fine-tune Output')
ax.legend()
plt.tight_layout()
plt.show()

# Breakdown
print(f'Distance = 0 (exact): {np.sum(distances == 0)}')
print(f'Distance 1-5 (near-miss): {np.sum((distances >= 1) & (distances <= 5))}')
print(f'Distance 6-20 (moderate): {np.sum((distances >= 6) & (distances <= 20))}')
print(f'Distance > 20 (far-off): {np.sum(distances > 20)}')

---
## 3. Over-generation Analysis

For prefix matches: how much extra content does the model generate after the correct part?

In [ ]:
prefix_data = []
for r in categorized:
    gt = r['gt_lower']
    out = r['out_lower']
    if not out or r['category'] == 'exact':
        continue
    # Find longest common prefix
    common = 0
    for a, b in zip(gt, out):
        if a == b:
            common += 1
        else:
            break
    if common > 3:  # meaningful prefix overlap
        excess = len(out) - common
        prefix_data.append({'common': common, 'excess': excess, 'gt_len': len(gt), 'out_len': len(out)})

if prefix_data:
    common_lens = [p['common'] for p in prefix_data]
    excess_lens = [p['excess'] for p in prefix_data]
    
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.scatter(common_lens, excess_lens, c=C_TEAL, alpha=0.6, s=40, edgecolor='#1a1a2e')
    ax.set_xlabel('Correct Prefix Length (chars)')
    ax.set_ylabel('Excess Characters After Prefix')
    ax.set_title(f'Over-generation: Correct Prefix vs Excess ({len(prefix_data)} clips with prefix overlap)')
    ax.axhline(0, color=C_DARK_TEAL, linestyle=':', alpha=0.5)
    plt.tight_layout()
    plt.show()
    
    print(f'Clips with prefix overlap > 3 chars: {len(prefix_data)}')
    print(f'Mean excess chars: {np.mean(excess_lens):.1f}')
    print(f'Max excess chars: {max(excess_lens)}')
else:
    print('No significant prefix overlaps found.')

---
## 4. Per-Word Accuracy

Which Dothraki words does the model get right most often? Which ones appear mainly in errors?

In [ ]:
from collections import Counter

correct_words = Counter()
error_words = Counter()

for r in categorized:
    gt_tokens = r['gt_lower'].split()
    out_tokens = set(r['out_lower'].split())
    
    for word in gt_tokens:
        if word in out_tokens:
            correct_words[word] += 1
        else:
            error_words[word] += 1

# Top 15 most correctly produced words
top_correct = correct_words.most_common(15)
# Top 15 most frequently missed words
top_errors = error_words.most_common(15)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

if top_correct:
    words, counts = zip(*top_correct)
    axes[0].barh(range(len(words)), counts, color=C_TEAL, edgecolor='#1a1a2e', alpha=0.85)
    axes[0].set_yticks(range(len(words)))
    axes[0].set_yticklabels(words, fontsize=10)
    axes[0].invert_yaxis()
    axes[0].set_xlabel('Occurrences')
    axes[0].set_title('Most Frequently Correct Words')

if top_errors:
    words, counts = zip(*top_errors)
    axes[1].barh(range(len(words)), counts, color=C_RED, edgecolor='#1a1a2e', alpha=0.85)
    axes[1].set_yticks(range(len(words)))
    axes[1].set_yticklabels(words, fontsize=10)
    axes[1].invert_yaxis()
    axes[1].set_xlabel('Occurrences')
    axes[1].set_title('Most Frequently Missed Words')

plt.tight_layout()
plt.show()

---
## 5. Length vs Accuracy

Are shorter utterances easier for the fine-tuned model? Bin clips by ground truth length and plot accuracy per bin.

In [ ]:
gt_lengths = [len(r['gt_dothraki']) for r in results]
exact_flags = [r.get('exact_match', False) for r in results]

# Bin by length
bins = [0, 15, 30, 50, 80, 200]
bin_labels = ['1-15', '16-30', '31-50', '51-80', '81+']
bin_correct = [0] * len(bin_labels)
bin_total = [0] * len(bin_labels)

for length, exact in zip(gt_lengths, exact_flags):
    for i in range(len(bins) - 1):
        if bins[i] < length <= bins[i + 1]:
            bin_total[i] += 1
            if exact:
                bin_correct[i] += 1
            break

bin_acc = [c / t * 100 if t > 0 else 0 for c, t in zip(bin_correct, bin_total)]

fig, ax = plt.subplots(figsize=(12, 6))
x = range(len(bin_labels))
bars = ax.bar(x, bin_acc, color=C_TEAL, edgecolor='#1a1a2e', alpha=0.85, width=0.6)

for i, bar in enumerate(bars):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{bin_acc[i]:.0f}%\n(n={bin_total[i]})', ha='center', va='bottom', fontsize=10)

ax.set_xticks(x)
ax.set_xticklabels([f'{l} chars' for l in bin_labels])
ax.set_ylabel('Exact Match Accuracy (%)')
ax.set_title('Fine-tune Accuracy by Ground Truth Length')
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

---
## 6. Example Gallery

A curated selection of 10 interesting cases showing the range of model behavior.

In [ ]:
# Select examples by category
examples = {'exact': [], 'prefix_match': [], 'wrong': [], 'empty': []}
for r in categorized:
    examples[r['category']].append(r)

# Also find near-misses (low Levenshtein distance but not exact)
near_misses = [(d, r) for d, r in zip(distances, categorized) if 1 <= d <= 5]
near_misses.sort(key=lambda x: x[0])

print('Example Gallery: Fine-tune Model Outputs')
print('=' * 90)

# 3 exact matches
print('\n--- EXACT MATCHES ---')
for r in examples['exact'][:3]:
    print(f'  [{r["id"]}] GT: {r["gt_dothraki"][:60]}')
    print(f'          Out: {r.get("raw_dothraki", "")[:60]}')
    print()

# 3 prefix matches (over-generation)
print('--- PREFIX MATCH (over-generation) ---')
for r in examples['prefix_match'][:3]:
    print(f'  [{r["id"]}] GT:  {r["gt_dothraki"][:60]}')
    print(f'          Out: {r.get("raw_dothraki", "")[:80]}')
    print()

# 2 wrong
print('--- ENTIRELY WRONG ---')
for r in examples['wrong'][:2]:
    print(f'  [{r["id"]}] GT:  {r["gt_dothraki"][:60]}')
    print(f'          Out: {r.get("raw_dothraki", "")[:60]}')
    print()

# 2 near-misses
print('--- NEAR-MISSES (edit distance 1-5) ---')
for dist, r in near_misses[:2]:
    print(f'  [{r["id"]}] GT:  {r["gt_dothraki"][:60]}')
    print(f'          Out: {r.get("raw_dothraki", "")[:60]}')
    print(f'          Edit distance: {dist}')
    print()

---
## Conclusions

1. **Over-generation is the dominant failure mode** — the model often starts correctly but fails to stop, generating excess tokens beyond the ground truth. This is consistent with the EOT (end-of-transcript) token training challenge.

2. **The model has learned Dothraki vocabulary** — per-word analysis shows it can produce many correct Dothraki tokens, suggesting the fine-tuning successfully taught vocabulary but not always sequence boundaries.

3. **Shorter utterances are easier** — accuracy drops with increasing ground truth length, confirming that the model struggles more with longer sequences where there's more opportunity for divergence.

4. **Many errors are near-misses** — a significant portion of non-exact matches have low edit distances, meaning the model is close but not quite right (punctuation, diacritics, extra characters).

**Key Takeaway:** The 48% exact match rate understates the model's capability — it has learned to produce Dothraki text but struggles with knowing when to stop. Better EOT training or length-conditioned decoding could improve results significantly.